In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=xvF9yS06ks4KhNVG0ehYZw5hfe7bY0&access_type=offline&code_challenge=hUVU_Mgwdr3Jvfp5XAX2ujjrB_JO8lHbF79movY7lh0&code_challenge_method=S256


Credentials saved to file: [/Users/yt4/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "open-targets-genetics-dev" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


Updates are available for some Google Clo

In [2]:
!gcloud auth login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=8JUW91DTFB6nQ4TMPGPbCBwkkeA1qj&access_type=offline&code_challenge=Sc3d6S95J90dj_QyrVGdr2r8j3yI1k7DvmFlIY3OOEo&code_challenge_method=S256


You are now logged in as [yt4@sanger.ac.uk].
Your current project is [open-targets-genetics-dev].  You can change this setting by running:
  $ gcloud config set project PROJECT_ID


In [1]:
import os

import hail as hl
import numpy as np
import pyspark.sql.functions as f
from pyspark.sql import DataFrame

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.dataset.study_locus import StudyLocus
from gentropy.susie_finemapper import SusieFineMapperStep
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment

Loading BokehJS ...

In [2]:
"""Common utilities for the project."""

import os
from pathlib import Path
from gentropy.common.session import Session
import logging


def get_gcs_credentials() -> str:
    """Get the credentials for google cloud storage."""
    app_default_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/application_default_credentials.json"
    )

    service_account_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/service_account_credentials.json"
    )

    if Path(app_default_credentials).exists():
        return app_default_credentials
    else:
        raise FileNotFoundError("No GCS credentials found.")


def get_gcs_hadoop_connector_jar() -> str:
    """Get the google cloud storage hadoop connector for spark.

    This function will return the url to download the hadoop jar.
    """

    return (
        "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar"
    )


def gcs_conf(
    credentials_path=None, project="open-targets-genetics-dev"
) -> dict[str, str]:
    """Get the spark configuration with hadoop connector for google cloud storage."""
    credentials_path = credentials_path or get_gcs_credentials()
    return {
        "spark.driver.memory": "12g",
        "spark.kryoserializer.buffer.max": "500m",
        "spark.driver.maxResultSize":"2g",
        "spark.hadoop.fs.gs.impl": "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        "spark.jars": get_gcs_hadoop_connector_jar(),
        "spark.hadoop.google.cloud.auth.service.account.enable": "true",
        "spark.hadoop.fs.gs.project.id": project,
        "spark.hadoop.google.cloud.auth.service.account.json.keyfile": credentials_path,
        "spark.hadoop.fs.gs.requester.pays.mode": "AUTO",
    }


class GentropySession(Session):
    def __init__(self, *args, **kwargs):
        if "extended_spark_conf" in kwargs:
            kwargs["extended_spark_conf"].update(gcs_conf())
        else:
            kwargs["extended_spark_conf"] = gcs_conf()
        super().__init__(*args, **kwargs)


session = GentropySession()

In [3]:
path_to_release_folder="gs://open-targets-data-releases/26.03/"

si=StudyIndex.from_parquet(session,path_to_release_folder+"output/study/")
sl=StudyLocus.from_parquet(session,path_to_release_folder+"output/credible_set/")

26/05/08 16:30:42 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@b7cdd4c{/SQL,null,AVAILABLE,@Spark}
26/05/08 16:30:42 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@3d8e142a{/SQL/json,null,AVAILABLE,@Spark}
26/05/08 16:30:42 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@21dec83a{/SQL/execution,null,AVAILABLE,@Spark}
26/05/08 16:30:42 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@76ea88f7{/SQL/execution/json,null,AVAILABLE,@Spark}
26/05/08 16:30:42 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@74efc1c5{/static/sql,null,AVAILABLE,@Spark}
26/05/08 16:30:44 INFO GhfsStorageStatistics: Detected potential high latency for operation op_get_file_status. latencyMs=589; previousMaxLatencyMs=0; operationCount=1; context=gs://open-targets-data-releases/26.03/output/study


# RARE VARINAT EVIDENCE

In [4]:
rare_variant_datasources = ["eva", "uniprot_variants", "gene2phenotype", "genomics_england", "clingen", "uniprot_literature", "orphanet"]

evidence = None
for datasource in rare_variant_datasources:
    path = f"{path_to_release_folder}output/evidence_{datasource}/"
    df = session.spark.read.parquet(path)
    if evidence is None:
        evidence = df
    else:
        evidence = evidence.unionByName(df, allowMissingColumns=True)

In [5]:
evidence = evidence.filter(f.col("score") >= 0.75).cache()
evidence.count()

26/05/08 16:30:59 INFO GhfsStorageStatistics: Detected potential high latency for operation stream_read_operations. latencyMs=503; previousMaxLatencyMs=487; operationCount=108; context=gs://open-targets-data-releases/26.03/output/evidence_eva/part-00019-d8931358-0756-4532-b02c-5f673ad428b7-c000.snappy.parquet
26/05/08 16:30:59 INFO GhfsStorageStatistics: Detected potential high latency for operation stream_read_operations. latencyMs=519; previousMaxLatencyMs=503; operationCount=114; context=gs://open-targets-data-releases/26.03/output/evidence_eva/part-00003-d8931358-0756-4532-b02c-5f673ad428b7-c000.snappy.parquet
26/05/08 16:30:59 INFO GhfsStorageStatistics: Detected potential high latency for operation stream_read_operations. latencyMs=558; previousMaxLatencyMs=519; operationCount=120; context=gs://open-targets-data-releases/26.03/output/evidence_eva/part-00006-d8931358-0756-4532-b02c-5f673ad428b7-c000.snappy.parquet
26/05/08 16:31:03 INFO GhfsStorageStatistics: Detected potential hi

459517

In [6]:
dd= evidence.groupBy("datasourceId").count()
dd.show()

+------------------+------+
|      datasourceId| count|
+------------------+------+
|               eva|360991|
|  uniprot_variants| 36120|
|    gene2phenotype|  3971|
|  genomics_england| 42455|
|           clingen|  2555|
|uniprot_literature|  6191|
|          orphanet|  7234|
+------------------+------+



In [7]:
evidence.show(1)

+---------------+--------------------+------------------+-------------------------+-------------+-------------------+---------------------+--------------------+--------------------+------------+-------------------+--------------------+-------------------+----------+-----------+------------+-------------------+------------------------------+--------------------+----------------+-----------+---------------+-------------+---------------+------------+-----+----------------+-----------------+----------------+-------------+----+----------------+
|       targetId|                  id|targetFromSourceId|diseaseFromSourceMappedId|alleleOrigins|allelicRequirements|clinicalSignificances|    cohortPhenotypes|          confidence|datasourceId|         datatypeId|   diseaseFromSource|diseaseFromSourceId|literature|releaseDate|     studyId|variantFromSourceId|variantFunctionalConsequenceId|       variantHgvsId|       variantId|variantRsId|qualityControls|    diseaseId|publicationDate|evidenceDate|sco

In [8]:
sgl0=evidence.select("diseaseId","targetId").distinct().cache()
sgl0.count()

29049

## CHEMBL

In [9]:
chembl=session.spark.read.parquet(path_to_release_folder+"/output/evidence_clinical_precedence/")

In [10]:
chembl.show(1)

+---------------+--------------------+------------------+-------------------------+--------------------+-------------+---------------+-------------------------+--------------+----------+--------------------+--------------------+------------+-------------------+----------+---------------+-----------+---------------+------------+-----+----------------+-----------------+
|       targetId|                  id|targetFromSourceId|diseaseFromSourceMappedId|    clinicalReportId|clinicalStage|trialWhyStopped|trialStopReasonCategories|studyStartDate|literature|   diseaseFromSource|      drugFromSource|      drugId|       datasourceId|datatypeId|qualityControls|  diseaseId|publicationDate|evidenceDate|score|directionOnTrait|directionOnTarget|
+---------------+--------------------+------------------+-------------------------+--------------------+-------------+---------------+-------------------------+--------------+----------+--------------------+--------------------+------------+-----------------

In [11]:
chembl.groupBy("score").count().show()

+-----+------+
|score| count|
+-----+------+
|  0.2|176318|
| 0.05|  8449|
|0.005|   318|
|  0.7|116235|
|  0.1| 74942|
|  1.0|129041|
|  0.8|    75|
| 0.35|  2145|
|  0.5| 14266|
|0.075|  1095|
| 0.15| 30648|
| 0.25|   263|
| 0.01| 45488|
|0.025|    61|
+-----+------+



In [12]:
chembl.groupBy("clinicalStage").count().show()

+-------------+------+
|clinicalStage| count|
+-------------+------+
|EARLY_PHASE_1|  6919|
|  PREAPPROVAL|    75|
|  PRECLINICAL|    24|
|   WITHDRAWAL|  1360|
|      UNKNOWN| 45782|
|    PHASE_2_3| 14387|
|     APPROVAL|102184|
|      PHASE_3|118380|
|      PHASE_4| 25639|
|    PHASE_1_2| 31743|
|      PHASE_1| 72167|
|      PHASE_2|180369|
|          IND|   315|
+-------------+------+



In [13]:
approved_phases=["PHASE_4","APPROVAL","PHASE_3","PREAPPROVAL"]

In [14]:
chembl4=chembl.filter((f.col("clinicalStage").isin(approved_phases)))
#chembl4=chembl4.select("diseaseId","targetId","clinicalStage","score","drugId").cache()
#chembl4.count()

In [15]:
sgl=chembl4.select("diseaseId","targetId").distinct().cache()
sgl.count()

25782

# Old GS

In [16]:
sgl2=session.spark.read.json("gs://genetics-portal-dev-analysis/yt4/2506_release/otg_gs_230511.json")

In [17]:
sgl2.printSchema()

root
 |-- association_info: struct (nullable = true)
 |    |-- otg_id: string (nullable = true)
 |-- gold_standard_info: struct (nullable = true)
 |    |-- gene_id: string (nullable = true)
 |    |-- highest_confidence: string (nullable = true)
 |-- metadata: struct (nullable = true)
 |    |-- set_label: string (nullable = true)
 |-- sentinel_variant: struct (nullable = true)
 |    |-- alleles: struct (nullable = true)
 |    |    |-- alternative: string (nullable = true)
 |    |    |-- reference: string (nullable = true)
 |    |-- locus_GRCh38: struct (nullable = true)
 |    |    |-- chromosome: string (nullable = true)
 |    |    |-- position: long (nullable = true)
 |-- trait_info: struct (nullable = true)
 |    |-- ontology: array (nullable = true)
 |    |    |-- element: string (containsNull = true)



In [18]:
sgl2.count()

1279

In [19]:
sgl2.show(1,truncate=False)

+----------------+-----------------------+-------------+-----------------------+---------------+
|association_info|gold_standard_info     |metadata     |sentinel_variant       |trait_info     |
+----------------+-----------------------+-------------+-----------------------+---------------+
|{GCST006061_3}  |{ENSG00000168356, High}|{ot_platform}|{{A, C}, {3, 38592651}}|{[EFO_0000275]}|
+----------------+-----------------------+-------------+-----------------------+---------------+
only showing top 1 row



In [20]:
sgl2.show(1,truncate=False)

+----------------+-----------------------+-------------+-----------------------+---------------+
|association_info|gold_standard_info     |metadata     |sentinel_variant       |trait_info     |
+----------------+-----------------------+-------------+-----------------------+---------------+
|{GCST006061_3}  |{ENSG00000168356, High}|{ot_platform}|{{A, C}, {3, 38592651}}|{[EFO_0000275]}|
+----------------+-----------------------+-------------+-----------------------+---------------+
only showing top 1 row



In [21]:
sgl2.groupBy("gold_standard_info.highest_confidence").count().show()

+------------------+-----+
|highest_confidence|count|
+------------------+-----+
|              High|  629|
|            Medium|  650|
+------------------+-----+



In [22]:
selected_df = sgl2.filter(f.col("gold_standard_info.highest_confidence").isin(["High","Medium"])).select( 
    f.col("gold_standard_info.gene_id").alias("geneId"),
    f.col("trait_info.ontology").alias("ontology")
)

# Show the results
selected_df.show(1,truncate=False)

+---------------+-------------+
|geneId         |ontology     |
+---------------+-------------+
|ENSG00000168356|[EFO_0000275]|
+---------------+-------------+
only showing top 1 row



In [23]:
exploded_df = selected_df.withColumn("efo_terms", f.explode("ontology")).drop("ontology")

exploded_df=exploded_df.withColumnRenamed("efo_terms","diseaseId").withColumnRenamed("geneId","targetId").select("diseaseId","targetId").distinct()
# Show the results
exploded_df.show(1,truncate=False)

+-------------+---------------+
|diseaseId    |targetId       |
+-------------+---------------+
|MONDO_0005090|ENSG00000140009|
+-------------+---------------+
only showing top 1 row



In [24]:
exploded_df.count()

812

# Combining

In [25]:
sgl.count()

25782

In [26]:
sgl.show(1)

+-----------+---------------+
|  diseaseId|       targetId|
+-----------+---------------+
|EFO_0000673|ENSG00000111816|
+-----------+---------------+
only showing top 1 row



In [27]:
exploded_df.count()

812

In [28]:
exploded_df.show(1)

+-------------+---------------+
|    diseaseId|       targetId|
+-------------+---------------+
|MONDO_0005090|ENSG00000140009|
+-------------+---------------+
only showing top 1 row



In [29]:
sgl0.count()

29049

In [30]:
sgl0.show(1)

+-------------+---------------+
|    diseaseId|       targetId|
+-------------+---------------+
|MONDO_0800130|ENSG00000185338|
+-------------+---------------+
only showing top 1 row



In [31]:
columns_order = exploded_df.columns

sgl_reordered = sgl.select(columns_order)
sgl0_reordered = sgl0.select(columns_order)
#shl_l_reordered = sgl_l.select(columns_order)


In [32]:
combined_df = exploded_df.union(sgl_reordered).union(sgl0_reordered).distinct().cache()
#combined_df = exploded_df.union(sgl_reordered).union(sgl0_reordered).union(shl_l_reordered).distinct().cache()
combined_df.show(1,truncate=False)

+-------------+---------------+
|diseaseId    |targetId       |
+-------------+---------------+
|MONDO_0005090|ENSG00000140009|
+-------------+---------------+
only showing top 1 row



In [33]:
combined_df.count()

55003

In [34]:
combined_df=combined_df.dropDuplicates(["targetId","diseaseId"]).cache()
combined_df.count()

55003

In [35]:
combined_df.write.mode("overwrite").parquet("2603_EGL_0.75_otg_chembl.parquet")